In [1]:
import pyspark
import os
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Kafka-Streaming').getOrCreate()


In [2]:
# Read Stream from Kafka

from pyspark.sql.functions import *

df_raw = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "localhost:9092")
         .option("subscribe", "uber-rides")
         .option("startingOffsets", "earliest")
         .load()
)

# Kafka gives key/value as binary → convert to string
df = df_raw.selectExpr("CAST(value AS STRING)")


Schema tells Spark the structure of the JSON message coming from Kafka. To properly parse JSON into columns, Spark must know what fields exist and what their data types are. from_json() parses the JSON string into a structured object (StructType)   something like: schema = StructType([
    StructField("ride_id", LongType()),
    StructField("city", StringType()),
    StructField("timestamp", StringType()),
    StructField("distance_km", DoubleType()),
    StructField("fare_usd", DoubleType())
])


.alias("data") 
Names the parsed struct column as data
data
   ride_id
   city
   timestamp
   distance_km
   fare_usd

.select("data.*")
This expands the struct into flat columns.
input in the form: data (struct<ride_id, city, timestamp, ...>)
output generated is:
| ride_id | city | timestamp  | distance_km | fare_usd |
| ------- | ---- | ---------- | ----------- | -------- |
| 101     | NYC  | 2025-01-01 | 4.3         | 12.5     |



In [7]:
schema = """
ride_id LONG,
city STRING,
timestamp STRING,
distance_km DOUBLE,
fare_usd DOUBLE
"""

df_parsed = df.select(from_json(col("value"), schema).alias("data")).select("data.*")


In [8]:
# get number of trips done for every minute

trips_per_min = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .groupBy(window(col("event_time"), "1 minute"))
        .count()
)


In [9]:
#Average ride price per city

#It computes average ride fare per city, over a 1-minute rolling event-time window, 
#from streaming Kafka data.
avg_fare = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .groupBy(
            window(col("event_time"), "1 minute"),
            col("city")
        )
        .agg(avg("fare_usd").alias("avg_fare"))
)


In [13]:
# Computing trips per minute

query1 = (
    trips_per_min.writeStream
        .format("console")
        .outputMode("complete")
        .option("truncate", False)
        .start()
)


In [14]:
# Computing average fare per city

query2 = (
    avg_fare.writeStream
        .format("console")
        .outputMode("complete")
        .option("truncate", False)
        .start()
)


In [15]:
query1.stop()
query2.stop()

In [16]:
import os

# Create folders for CSV text files
os.makedirs("stored_csv/trips_per_min", exist_ok=True)
os.makedirs("stored_csv/avg_fare", exist_ok=True)

# Create folders for progress tracking
os.makedirs("checkpoints/csv_trips_checkpoint", exist_ok=True)
os.makedirs("checkpoints/csv_fare_checkpoint", exist_ok=True)

print("CSV storage folders ready!")


CSV storage folders ready!


In [19]:
# 1. Stop background streams
for s in spark.streams.active:
    s.stop()

# 2. Start query1 using memory sink + complete mode
query1 = (
    trips_per_min.writeStream
        .format("memory")
        .queryName("trips_table")
        .outputMode("complete") # Works perfectly here!
        .start()
)

# 3. Start query2 using memory sink + complete mode
query2 = (
    avg_fare.writeStream
        .format("memory")
        .queryName("fare_table")
        .outputMode("complete") # Works perfectly here!
        .start()
)

print("Streams started successfully in COMPLETE mode!")


Streams started successfully in COMPLETE mode!


In [20]:
# View the running totals inside memory
spark.sql("SELECT * FROM trips_table").show(truncate=False)
spark.sql("SELECT * FROM fare_table").show(truncate=False)


+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2026-09-16 15:07:00, 2026-09-16 15:08:00}|60   |
|{2026-09-16 15:52:00, 2026-09-16 15:53:00}|60   |
|{2026-09-16 15:28:00, 2026-09-16 15:29:00}|60   |
|{2026-09-16 16:25:00, 2026-09-16 16:26:00}|60   |
|{2026-09-16 15:26:00, 2026-09-16 15:27:00}|60   |
|{2026-09-16 15:20:00, 2026-09-16 15:21:00}|60   |
|{2026-09-16 15:53:00, 2026-09-16 15:54:00}|60   |
|{2026-09-16 15:46:00, 2026-09-16 15:47:00}|60   |
|{2026-09-16 15:27:00, 2026-09-16 15:28:00}|60   |
|{2026-09-16 15:55:00, 2026-09-16 15:56:00}|60   |
|{2026-09-16 16:22:00, 2026-09-16 16:23:00}|60   |
|{2026-09-16 15:08:00, 2026-09-16 15:09:00}|60   |
|{2026-09-16 15:12:00, 2026-09-16 15:13:00}|28   |
|{2026-09-16 15:22:00, 2026-09-16 15:23:00}|60   |
|{2026-09-16 15:50:00, 2026-09-16 15:51:00}|59   |
|{2026-09-16 15:30:00, 2026-09-16 15:31:00}|59   |
|{2026-09-16 15:56:00, 2026-09-

In [21]:
# 1. Stop any active background streams
for s in spark.streams.active:
    s.stop()

# 2. Print Trips Per Minute directly to the console
query1 = (
    trips_per_min.writeStream
        .format("console")        # <-- Directs output to console
        .outputMode("complete")   # <-- Keeps complete mode
        .option("truncate", False)
        .start()
)

# 3. Print Average Fare Per City directly to the console
query2 = (
    avg_fare.writeStream
        .format("console")        # <-- Directs output to console
        .outputMode("complete")   # <-- Keeps complete mode
        .option("truncate", False)
        .start()
)

print("Streaming queries started! Check your VS Code terminal for the console tables.")




Streaming queries started! Check your VS Code terminal for the console tables.


In [22]:
# UPDATE YOUR TRIPS AGGREGATION CELL:
trips_per_min = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .withWatermark("event_time", "10 minutes")  # <-- ADD THIS LINE
        .groupBy(window(col("event_time"), "1 minute"))
        .count()
)

In [23]:
avg_fare = (
    df_parsed
        .withColumn("event_time", to_timestamp("timestamp"))
        .withWatermark("event_time", "10 minutes")  # <-- ADD THIS LINE
        .groupBy(
            window(col("event_time"), "1 minute"),
            col("city")
        )
        .agg(avg("fare_usd").alias("avg_fare"))
)

In [27]:
# 1. Stop any background streams using the correct .active property
for s in spark.streams.active:  # <-- Added .active here
    s.stop()

# 2. Flatten the window column into a string for CSV compatibility
trips_csv_ready = trips_per_min.select(col("window").cast("string").alias("time_window"), "count")
avg_fare_csv_ready = avg_fare.select(col("window").cast("string").alias("time_window"), "city", "avg_fare")

# 3. Save Trips Per Minute to CSV
query1 = (
    trips_csv_ready.writeStream
        .format("csv")
        .outputMode("append")
        .option("header", "true") 
        .option("path", "stored_csv/trips_per_min")
        .option("checkpointLocation", "checkpoints/csv_trips_checkpoint")
        .start()
)

# 4. Save Average Fare Per City to CSV
query2 = (
    avg_fare_csv_ready.writeStream
        .format("csv")
        .outputMode("append")
        .option("header", "true")
        .option("path", "stored_csv/avg_fare")  # Kept checkpoint unique
        .option("checkpointLocation", "checkpoints/csv_fare_checkpoint")
        .start()
)

print("Streaming data is now successfully writing directly to CSV files!")


Streaming data is now successfully writing directly to CSV files!


In [25]:
query1.stop()

In [26]:
query2.stop()